# indian_equities_1h_strat_003

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (142).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `indian_equities` |
| Trading style | `unknown` |
| Timeframe | `1h` |
| Code cells | 9 |
| Detected functions | halftrend, halftrend_with_session_reset, backtest_halftrend, _run_group, backtest_multiple_tickers |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/NIFTY50_AUG_OCT_25.csv")
df

In [ ]:
df["Ticker"].unique()

In [ ]:
import pandas as pd

# =========================================
# Step 1: Preprocess Data
# =========================================
df['Date_Time'] = pd.to_datetime(df['Date_Time'])
df = (
    df.set_index('Date_Time')
      .dropna()
      .sort_index()[["Ticker", "open", "high", "low", "close"]]
)

# =========================================
# Step 2: Resample to Hourly (with 15-min offset)
# =========================================
df = (
    df.groupby("Ticker", group_keys=False)
      .resample("1h", offset="15min")    # hourly bars, starting at 09:15
      .agg({
          "open": "first",
          "high": "max",
          "low": "min",
          "close": "last",
          "Ticker": "first" # Keep Ticker column after aggregation
      })
      .dropna()
)

# =========================================
# Final Dataset
# =========================================
df

In [ ]:
df = df[df['Ticker'].isin(["BEL.NSE", "TRENT.NSE", "LT.NSE", "COALINDIA.NSE", "BPCL.NSE",
                           "GRASIM.NSE", "POWERGRID.NSE", "RELIANCE.NSE", "ONGC.NSE",
                           "ULTRACEMCO.NSE", "NTPC.NSE"])]
df

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# HalfTrend Indicator
# ============================================================

def halftrend(df: pd.DataFrame,
              amplitude: int = 2,
              channel_deviation: int = 2,
              atr_period: int = 100) -> pd.DataFrame:
    """
    HalfTrend indicator for a single continuous time series (one ticker).
    """
    df = df.copy()

    high, low, close = df['high'].values, df['low'].values, df['close'].values
    n = len(df)

    # --- ATR (EMA) ---------------------------------------------------------
    hl_range = high - low
    hc_range = np.abs(high - np.roll(close, 1))
    lc_range = np.abs(low - np.roll(close, 1))
    true_range = np.maximum(hl_range, np.maximum(hc_range, lc_range))
    true_range[0] = hl_range[0]

    atr = np.zeros(n)
    alpha = 2.0 / (atr_period + 1)
    atr[0] = true_range[0]
    for i in range(1, n):
        atr[i] = alpha * true_range[i] + (1 - alpha) * atr[i - 1]

    dev = channel_deviation * (atr / 2)

    # --- Arrays for state --------------------------------------------------
    trend = np.zeros(n, dtype=int)
    next_trend = np.zeros(n, dtype=int)
    max_low_price = np.zeros(n)
    min_high_price = np.zeros(n)
    up, down, ht = np.zeros(n), np.zeros(n), np.zeros(n)

    # --- Initialize first bar ----------------------------------------------
    trend[0] = next_trend[0] = 0
    max_low_price[0], min_high_price[0] = low[0], high[0]
    up[0], down[0], ht[0] = low[0], high[0], close[0]

    # --- Main loop ---------------------------------------------------------
    for i in range(1, n):
        start = max(0, i - amplitude)
        high_price = np.max(high[start:i + 1])
        low_price = np.min(low[start:i + 1])

        high_ma = np.mean(high[i - amplitude + 1:i + 1]) if i >= amplitude else np.mean(high[:i + 1])
        low_ma = np.mean(low[i - amplitude + 1:i + 1]) if i >= amplitude else np.mean(low[:i + 1])

        if next_trend[i - 1] == 0:  # expecting uptrend
            max_low_price[i] = max(low_price, max_low_price[i - 1])
            if high_ma < max_low_price[i] and close[i] < low[i - 1]:
                trend[i] = next_trend[i] = 1
                min_high_price[i] = high_price
            else:
                trend[i] = trend[i - 1]
                next_trend[i] = next_trend[i - 1]
                min_high_price[i] = min_high_price[i - 1]
        else:  # expecting downtrend
            min_high_price[i] = min(high_price, min_high_price[i - 1])
            if low_ma > min_high_price[i] and close[i] > high[i - 1]:
                trend[i] = next_trend[i] = 0
                max_low_price[i] = low_price
            else:
                trend[i] = trend[i - 1]
                next_trend[i] = next_trend[i - 1]
                max_low_price[i] = max_low_price[i - 1]

        if trend[i] == 0:  # uptrend
            up[i] = down[i - 1] if trend[i - 1] == 1 else max(max_low_price[i], up[i - 1])
            down[i] = down[i - 1]
            ht[i] = up[i]
        else:  # downtrend
            down[i] = up[i - 1] if trend[i - 1] == 0 else min(min_high_price[i], down[i - 1])
            up[i] = up[i - 1]
            ht[i] = down[i]

    # --- Signals -----------------------------------------------------------
    signal = np.zeros(n, dtype=int)
    for i in range(1, n):
        if trend[i-1] == 1 and trend[i] == 0:
            signal[i] = 1   # Buy
        elif trend[i-1] == 0 and trend[i] == 1:
            signal[i] = -1  # Sell

    # --- Output ------------------------------------------------------------
    df['HalfTrend'] = ht
    df['ATR_High'] = ht + dev
    df['ATR_Low'] = ht - dev
    df['Signal'] = signal
    df['Trend'] = trend
    df['ATR'] = atr

    return df


# ============================================================
# Wrapper for Multi-Ticker, Daily Session Reset (index-based)
# ============================================================

def halftrend_with_session_reset(df: pd.DataFrame,
                                 amplitude: int = 2,
                                 channel_deviation: int = 2,
                                 atr_period: int = 100) -> pd.DataFrame:
    """
    Apply HalfTrend day by day per ticker to avoid ATR/trend leakage.
    Assumes 'Date_Time' is the index.
    Requires columns: ['Ticker','open','high','low','close'].
    """
    df = df.copy()
    df['date_only'] = df.index.date  # extract date from index

    results = []
    for (ticker, d), group in df.groupby(['Ticker', 'date_only']):
        results.append(halftrend(group, amplitude, channel_deviation, atr_period))

    return (
        pd.concat(results)
          .sort_values(["Ticker", "Date_Time"])
          .drop(columns=['date_only'])
    )

# ============================================================
# Example Usage
# ============================================================

data = halftrend_with_session_reset(df, amplitude=3, channel_deviation=2, atr_period=100)
data


In [ ]:
data["Signal"].unique()

In [ ]:
import numpy as np
import pandas as pd
from typing import Tuple, Optional

# ============================================================
# Backtester with HalfTrend + ATR Trailing SL
# ============================================================

def backtest_halftrend(
    data: pd.DataFrame,
    amplitude: int = 2,
    channel_deviation: int = 2,
    atr_period: int = 100,
    atr_multiplier: float = 1.5,        # k in the formula: New SL = max(Old SL, Current Price - ATR × k)
    capital_per_trade: float = 100_000.0,
    slippage_bps: float = 5.0,          # 5 bps = 0.05% each side
    brokerage_rate: float = 0.001,      # 0.10% per side
    use_next_open: bool = True,         # fill on next bar open
    time_col: str = None                # optional datetime column
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Backtester for HalfTrend with ATR-based trailing SL.

    Entry: HalfTrend reversal signal.
    Exit: Opposite signal OR ATR-based trailing SL OR EOD force exit.

    Trailing SL Formula: New SL = max(Old SL, Current Price - ATR × k)

    Returns:
      trades_df: Tradebook with detailed logs
      out_df: Data with indicators and signals (for all rows)
    """
    df = data.copy()

    # Ensure datetime index
    if time_col and time_col in df.columns:
        df[time_col] = pd.to_datetime(df[time_col])
        df = df.sort_values([time_col] + (["Ticker"] if "Ticker" in df.columns else []))
        df = df.set_index(time_col)
    else:
        if not isinstance(df.index, pd.DatetimeIndex):
            raise ValueError("Provide a datetime index or specify time_col.")
        df = df.sort_index()
        if "Ticker" in df.columns:
            df = df.sort_values(["Ticker"]).sort_index()

    # Apply HalfTrend indicator
    if 'Ticker' in df.columns:
        out_df = df.groupby('Ticker', group_keys=False).apply(
            lambda x: halftrend(x, amplitude, channel_deviation, atr_period)
        )
    else:
        out_df = halftrend(df, amplitude, channel_deviation, atr_period)

    # Helper to run per ticker
    def _run_group(g: pd.DataFrame, tkr: Optional[str]) -> list:
        g = g.copy()
        if use_next_open:
            exec_price = g['open'].shift(-1)
        else:
            exec_price = g['close']

        g['exec_price'] = exec_price
        g = g.dropna(subset=['exec_price'])

        slips = slippage_bps / 10000.0
        trades = []
        position = 0  # +1 long, -1 short, 0 flat
        entry_price = None
        entry_time = None
        qty = 0
        trailing_sl = None  # Track the trailing stop loss level

        for ts, row in g.iterrows():
            sig = int(row['Signal'])

            # ============================================================
            # TRAILING STOP LOSS LOGIC: New SL = max(Old SL, Current Price - ATR × k)
            # ============================================================
            if position == 1:  # long position
                current_price = row['close']  # Use close price for SL calculation
                atr_value = row.get('ATR', 0)  # Get ATR value

                # Calculate new trailing SL: max(Old SL, Current Price - ATR × k)
                new_sl = current_price - (atr_value * atr_multiplier)

                if trailing_sl is None:
                    trailing_sl = new_sl
                else:
                    trailing_sl = max(trailing_sl, new_sl)

                # Check if stop loss is hit (low of the bar falls below SL)
                if row['low'] <= trailing_sl:
                    px = row['exec_price']
                    exit_px = px * (1 - slips)
                    gross = qty * (exit_px - entry_price)
                    notional_exit = qty * exit_px
                    notional_entry = qty * entry_price
                    cost = brokerage_rate * (notional_entry + notional_exit)
                    pnl = gross - cost
                    trades.append({
                        'Ticker': tkr if tkr is not None else '',
                        'Entry Time': entry_time,
                        'Exit Time': ts,
                        'Action': 'Long SL Hit',
                        'Entry Price': entry_price,
                        'Exit Price': exit_px,
                        'Quantity': qty,
                        'Gross PnL': gross,
                        'Brokerage': cost,
                        'Net PnL': pnl,
                        'SL Level': trailing_sl
                    })
                    position, qty, entry_price, entry_time, trailing_sl = 0, 0, None, None, None
                    continue  # move to next bar

            elif position == -1:  # short position
                current_price = row['close']  # Use close price for SL calculation
                atr_value = row.get('ATR', 0)  # Get ATR value

                # For short positions: New SL = min(Old SL, Current Price + ATR × k)
                new_sl = current_price + (atr_value * atr_multiplier)

                if trailing_sl is None:
                    trailing_sl = new_sl
                else:
                    trailing_sl = min(trailing_sl, new_sl)

                # Check if stop loss is hit (high of the bar rises above SL)
                if row['high'] >= trailing_sl:
                    px = row['exec_price']
                    exit_px = px * (1 + slips)
                    gross = qty * (entry_price - exit_px)
                    notional_exit = qty * exit_px
                    notional_entry = qty * entry_price
                    cost = brokerage_rate * (notional_entry + notional_exit)
                    pnl = gross - cost
                    trades.append({
                        'Ticker': tkr if tkr is not None else '',
                        'Entry Time': entry_time,
                        'Exit Time': ts,
                        'Action': 'Short SL Hit',
                        'Entry Price': entry_price,
                        'Exit Price': exit_px,
                        'Quantity': qty,
                        'Gross PnL': gross,
                        'Brokerage': cost,
                        'Net PnL': pnl,
                        'SL Level': trailing_sl
                    })
                    position, qty, entry_price, entry_time, trailing_sl = 0, 0, None, None, None
                    continue  # move to next bar

            # ============================================================
            # SIGNAL-BASED ENTRY/EXIT
            # ============================================================
            desired = position
            if sig == 1:
                desired = 1
            elif sig == -1:
                desired = -1

            if desired != position:
                px = row['exec_price']

                # Close short -> enter long
                if desired == 1:
                    if position == -1 and qty > 0:
                        exit_px = px * (1 + slips)
                        gross = qty * (entry_price - exit_px)
                        notional_exit = qty * exit_px
                        notional_entry = qty * entry_price
                        cost = brokerage_rate * (notional_entry + notional_exit)
                        pnl = gross - cost
                        trades.append({
                            'Ticker': tkr if tkr is not None else '',
                            'Entry Time': entry_time,
                            'Exit Time': ts,
                            'Action': 'Sell->Cover',
                            'Entry Price': entry_price,
                            'Exit Price': exit_px,
                            'Quantity': qty,
                            'Gross PnL': gross,
                            'Brokerage': cost,
                            'Net PnL': pnl,
                            'SL Level': trailing_sl
                        })
                        position, qty, entry_price, trailing_sl = 0, 0, None, None

                    buy_px = px * (1 + slips)
                    qty = int(np.floor(capital_per_trade / buy_px))
                    if qty > 0:
                        position, entry_price, entry_time = 1, buy_px, ts
                        # Initialize trailing SL for new long position
                        current_price = row['close']
                        atr_value = row.get('ATR', 0)
                        trailing_sl = current_price - (atr_value * atr_multiplier)

                # Close long -> enter short
                elif desired == -1:
                    if position == 1 and qty > 0:
                        exit_px = px * (1 - slips)
                        gross = qty * (exit_px - entry_price)
                        notional_exit = qty * exit_px
                        notional_entry = qty * entry_price
                        cost = brokerage_rate * (notional_entry + notional_exit)
                        pnl = gross - cost
                        trades.append({
                            'Ticker': tkr if tkr is not None else '',
                            'Entry Time': entry_time,
                            'Exit Time': ts,
                            'Action': 'Buy->Sell',
                            'Entry Price': entry_price,
                            'Exit Price': exit_px,
                            'Quantity': qty,
                            'Gross PnL': gross,
                            'Brokerage': cost,
                            'Net PnL': pnl,
                            'SL Level': trailing_sl
                        })
                        position, qty, entry_price, trailing_sl = 0, 0, None, None

                    sell_px = px * (1 - slips)
                    qty = int(np.floor(capital_per_trade / sell_px))
                    if qty > 0:
                        position, entry_price, entry_time = -1, sell_px, ts
                        # Initialize trailing SL for new short position
                        current_price = row['close']
                        atr_value = row.get('ATR', 0)
                        trailing_sl = current_price + (atr_value * atr_multiplier)

        # ============================================================
        # EOD FORCE EXIT
        # ============================================================
        if position != 0 and qty > 0:
            ts = g.index[-1]
            px = g.iloc[-1]['exec_price']
            if position == 1:
                exit_px = px * (1 - slips)
                gross = qty * (exit_px - entry_price)
                notional_exit = qty * exit_px
                notional_entry = qty * entry_price
                cost = brokerage_rate * (notional_entry + notional_exit)
                pnl = gross - cost
                trades.append({
                    'Ticker': tkr if tkr is not None else '',
                    'Entry Time': entry_time,
                    'Exit Time': ts,
                    'Action': 'Buy->Sell(EOD)',
                    'Entry Price': entry_price,
                    'Exit Price': exit_px,
                    'Quantity': qty,
                    'Gross PnL': gross,
                    'Brokerage': cost,
                    'Net PnL': pnl,
                    'SL Level': trailing_sl
                })
            else:
                exit_px = px * (1 + slips)
                gross = qty * (entry_price - exit_px)
                notional_exit = qty * exit_px
                notional_entry = qty * entry_price
                cost = brokerage_rate * (notional_entry + notional_exit)
                pnl = gross - cost
                trades.append({
                    'Ticker': tkr if tkr is not None else '',
                    'Entry Time': entry_time,
                    'Exit Time': ts,
                    'Action': 'Sell->Cover(EOD)',
                    'Entry Price': entry_price,
                    'Exit Price': exit_px,
                    'Quantity': qty,
                    'Gross PnL': gross,
                    'Brokerage': cost,
                    'Net PnL': pnl,
                    'SL Level': trailing_sl
                })

        return trades

    # ============================================================
    # Run per ticker or whole dataset
    # ============================================================
    all_trades = []
    if 'Ticker' in out_df.columns:
        for tkr, g in out_df.groupby('Ticker'):
            all_trades.extend(_run_group(g, tkr))
    else:
        all_trades.extend(_run_group(out_df, None))

    trades_df = pd.DataFrame(all_trades)
    if not trades_df.empty:
        trades_df = trades_df.sort_values('Exit Time').reset_index(drop=True)
        trades_df['Transaction Cost'] = trades_df['Brokerage']
        trades_df['PnL'] = trades_df['Net PnL']
        trades_df['Cumulative PnL'] = trades_df['PnL'].cumsum()
        trades_df['Portfolio Value'] = 100_000.0 + trades_df['Cumulative PnL']

    return trades_df, out_df

In [ ]:
def backtest_multiple_tickers(
    data: pd.DataFrame,
    tickers: list,
    amplitude: int = 2,
    channel_deviation: int = 2,
    atr_period: int = 100,
    capital_per_trade: float = 100_000.0,
    slippage_bps: float = 5.0,
    brokerage_rate: float = 0.001,
    use_next_open: bool = True,
    time_col: str = 'Date_Time'
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Backtest multiple tickers and return combined tradebook and indicator DataFrame.

    Returns:
        combined_trades_df: All trades for all tickers
        combined_data_df: Original data with indicators and signals
    """
    all_trades = []
    all_data = []

    for tkr in tickers:
        ticker_data = data[data['Ticker'] == tkr].copy()
        if ticker_data.empty:
            continue

        trades_df, data_with_ind = backtest_halftrend(
            ticker_data,
            amplitude=amplitude,
            channel_deviation=channel_deviation,
            atr_period=atr_period,
            capital_per_trade=capital_per_trade,
            slippage_bps=slippage_bps,
            brokerage_rate=brokerage_rate,
            use_next_open=use_next_open,
            time_col=time_col
        )

        all_trades.append(trades_df)
        all_data.append(data_with_ind)

    combined_trades_df = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    combined_data_df = pd.concat(all_data, ignore_index=False) if all_data else pd.DataFrame()

    return combined_trades_df, combined_data_df


# ================================
# Example usage
# ================================
tickers_list = data['Ticker'].unique().tolist()
all_trades, all_data_with_ind = backtest_multiple_tickers(
    data,
    tickers_list,
    amplitude=3,
    channel_deviation=2,
    atr_period=18,
    capital_per_trade=100_000.0,
    slippage_bps=5.0,
    brokerage_rate=0.001,
    use_next_open=True,
    time_col='Date_Time'
)

# Save tradebook
all_trades.to_csv('/content/drive/MyDrive/tradebook_all_tickers.csv', index=False)
all_trades